# Stan Radon Random Restarts

Runs Stan mean-field ADVI random restarts for the PyMC-tutorial varying-intercept radon model.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/radon_example/pymc_tutorial")
STAN_FILE_NAME = "stan_radon_varying_intercept.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


In [ ]:
from modulars.radon import (
    default_radon_plot_dims,
    load_best_references,
    load_radon_data,
    make_radon_unconstrained_param_names,
    plot_radon_selected_dims,
    plot_radon_selected_dims_zoomed,
    print_reference_summary,
)

radon_data = load_radon_data()
county_names = radon_data["county_names"]
param_names = make_radon_unconstrained_param_names(county_names)
dim = len(param_names)
which_dims, which_labels = default_radon_plot_dims(param_names, county_names)

stan_data = {
    "N": int(len(radon_data["log_radon"])),
    "C": int(len(county_names)),
    "log_radon": np.asarray(radon_data["log_radon"], dtype=float),
    "floor": np.asarray(radon_data["floor"], dtype=float),
    "county": np.asarray(radon_data["county"], dtype=int) + 1,
}
PARAM_COLUMNS = (
    ["mu_a", "sigma_a"]
    + [f"alpha[{i}]" for i in range(1, len(county_names) + 1)]
    + ["beta", "sd_y"]
)
LOG_COLUMNS = ["sigma_a", "sd_y"]

print(f"N observations: {stan_data['N']}")
print(f"N counties: {stan_data['C']}")
print(f"Latent VI dimension: {dim}")
print(list(zip(which_dims, which_labels)))


In [ ]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 50, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "results"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


In [ ]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_tuple

stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=results_dir / "cmdstan_runs",
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
)

single_means, single_stds, multi_means, multi_stds = stan_result_tuple(stan_result)
iterations = stan_result["iterations"]
print(single_means.shape, single_stds.shape, multi_means.shape, multi_stds.shape)
print(f"tracked iterations: {iterations[:5]} ... {iterations[-5:]}")
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")


In [ ]:
from modulars import save_to_csv

save_to_csv(
    results_dir / "stan_processed_restarts.csv",
    [(single_means, single_stds, multi_means, multi_stds)],
)

save_to_csv(
    results_dir / "stan_final_restarts.csv",
    [(multi_means[:, -1, :], multi_stds[:, -1, :])],
)

results_dir


In [ ]:
from modulars import load_from_csv

print_reference_summary()
reference_summary, best_reference, reference_means, reference_stds = load_best_references()

single_means, single_stds, multi_means, multi_stds = load_from_csv(
    results_dir / "stan_processed_restarts.csv"
)[0]

plot_radon_selected_dims_zoomed(
    single_means,
    single_stds,
    multi_means,
    multi_stds,
    which_dims,
    which_labels,
    title_prefix="Stan ",
    reference_means=reference_means,
    reference_stds=reference_stds,
    x=iterations,
)
plot_radon_selected_dims(
    single_means,
    single_stds,
    multi_means,
    multi_stds,
    which_dims,
    which_labels,
    title_prefix="Stan ",
    reference_means=reference_means,
    reference_stds=reference_stds,
    x=iterations,
)
